# Email dataset — quick look

A labeled email-classification corpus (**378,616 emails × 32 columns**) assembled from
several public sources. Each row is one email: its raw text plus engineered features
used for phishing detection.

- **Labels:** legitimate ~82%, phishing ~17%, spam <1%.
- **Sources:** mostly the **Enron** corporate corpus (legitimate), plus phishing corpora
  (**Nazario**, **phishing_pot**, `meajor`, etc.).
- **Features:** URL counts, IP-in-URL flag, HTML flag, attachment counts, SPF result, and an
  `urgency_level` label (phishing skews more urgent than legitimate).
- Mostly early-2000s mail (Enron era) with a recent 2023–2026 slice.
- Note: `sender`/`subject` are often null on Enron rows — that metadata lives inside
  `email_text` / `headers`.

In [1]:
import os
import pandas as pd

pd.set_option("display.max_colwidth", 80)

# Prefer the full local parquet; fall back to the 100-row sample, then to a live fetch.
candidates = [
    "../data/external/normalized_dataset_full.parquet",
    "../data/external/normalized_dataset_sample100.parquet",
]
path = next((p for p in candidates if os.path.exists(p)), None)
if path:
    df = pd.read_parquet(path)
    print("loaded:", path)
else:
    import redivis
    t = (redivis.user("zw3155")
         .dataset("coms6901_email_dataset_original_repo_output:e83c:v1_0")
         .table("normalized_dataset:a8hm"))
    df = t.to_pandas_dataframe(max_results=100)
    print("loaded via redivis (100-row sample)")

print("shape:", df.shape)

loaded: ../data/external/normalized_dataset_full.parquet
shape: (378616, 32)


## Column names

In [2]:
for i, (name, dtype) in enumerate(df.dtypes.items(), 1):
    print(f"{i:>2}. {name:<24} {dtype}")

 1. email_text               string
 2. subject                  string
 3. sender                   string
 4. sender_domain            string
 5. receiver                 string
 6. receiver_domain          string
 7. date                     string
 8. normalized_label         string
 9. label_id                 double[pyarrow]
10. source                   string
11. year                     double[pyarrow]
12. num_urls                 int64[pyarrow]
13. has_ip_url               int64[pyarrow]
14. email_length             int64[pyarrow]
15. num_exclamation_marks    int64[pyarrow]
16. num_links_in_body        int64[pyarrow]
17. is_html_email            int64[pyarrow]
18. url_domains              string
19. url_length_max           double[pyarrow]
20. url_length_avg           double[pyarrow]
21. url_subdom_max           double[pyarrow]
22. url_subdom_avg           double[pyarrow]
23. attachment_count         double[pyarrow]
24. has_attachments          string
25. content_types        

## What kind of emails (label & source mix)

In [3]:
print("normalized_label:")
print(df["normalized_label"].value_counts(dropna=False).to_string())
print("\nsource:")
print(df["source"].value_counts(dropna=False).to_string())

normalized_label:
normalized_label
legitimate    311672
phishing       65954
spam             990

source:
source
enron                 243116
meajor                105920
phishing_pot           12027
rpuv_email_dataset     10308
nazario                 2755
github                  2678
nazario_monkey          1756
scraped_spam              56


## Five example emails\n\nTwo phishing, two legitimate, one spam (fixed seed, body truncated).

In [4]:
def pick(label, n):
    sub = df[df["normalized_label"] == label]
    return sub.sample(min(n, len(sub)), random_state=0) if len(sub) else sub

examples = pd.concat([pick("phishing", 2), pick("legitimate", 2), pick("spam", 1)])

for i, (_, r) in enumerate(examples.iterrows(), 1):
    print("=" * 90)
    print(f"Email {i}: {r['normalized_label'].upper()}  "
          f"(source={r['source']}, urls={r['num_urls']}, urgency={r['urgency_level']})")
    print(f"subject: {r['subject']}")
    print(f"sender : {r['sender']}")
    body = str(r["email_text"])
    print("-" * 90)
    print(body[:600] + ("..." if len(body) > 600 else ""))
    print()

Email 1: PHISHING  (source=nazario_monkey, urls=1, urgency=not urgent)
subject: monkey.org Server - Password Expired 
sender : Server Admin - monkey.org info@exalplast.com
------------------------------------------------------------------------------------------
monkey.org Server - Password Expired The password to your mailbox jose@monkey.org mailto:jose@monkey.org has expired. System will log you out and generate a new password exactly at 24 hours from 5/2/2023 7:38:51 a.m. . You can continue using your current password. Use the button below to keep using current password. Keep Current Password https://pub-801a7cdc811a4c008ff1aa59aa83c5d3.r2.dev/index2.html#jose@monkey.org Email is generated by monkey.org Email Server for jose@monkey.org mailto:jose@monkey.org

Email 2: PHISHING  (source=phishing_pot, urls=2, urgency=somewhat urgent)
subject: Prachtige =?utf-8?Q?Oekra=C3=AFense?= Meisjes Wachten om Jou te
 Ontmoeten op Valentijn!
sender : Vind =?utf-8?Q?Oekra=C3=AFense?= Liefde info@m